# Run and monitor Experiment 4A, then 4B (shared adapter)

Same background/log-tail pattern as `run_exp4.ipynb`, adapted for the two-step plan:

1. **4A**: trains a CBT-DP-DPO adapter on the TRAIN split only (124 exercises), tests it in
   English on the 32 held-out test exercises.
2. **4B**: reuses that SAME adapter (no retraining -- `SKIP_TRAIN=1`) to test on Arabic
   Shifaa questions. Valid because Shifaa is unrelated content to any CBT-Bench exercise,
   so training on 124 vs 156 exercises doesn't affect 4B's validity, but DOES matter for
   4A (testing on held-out English exercises the model must not have trained on).

**Run cells top to bottom for 4A first.** Only move to the 4B section once 4A's log shows
`Done. Summary written to ...` -- 4B's launch cell will refuse to run otherwise.

**Before opening this**, make sure the venv is set up in your terminal at least once this
session (module loads don't persist across sessions, only the venv files do):
```bash
cd /SEAS/home/<your-netid>/dpo/mh-dpo-master
source setup_env.sh
jupyter lab --no-browser --port=8888 --ip=0.0.0.0
```
then tunnel from your laptop: `ssh -L 8888:<gpu-node>:8888 <your-netid>@pegasus.arc.gwu.edu`
(swap `<gpu-node>` for whichever GPU node you're actually on).

## 0. Sanity checks

In [ ]:
import subprocess

def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True)

sh("pwd")
sh("ls run_exp_4.sh run_exp_4a.sh src/judge.py src/generate.py")
sh("echo VIRTUAL_ENV=$VIRTUAL_ENV")
sh("python3 -c \"import torch, transformers, peft, trl, datasets, accelerate; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())\"")
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")

---
# Part 1: Experiment 4A (train on TRAIN split, test in English)

## 1A. Configure

In [ ]:
RUN_TAG = "trainsplit"
SKILL_WEIGHTS = '{"SQ": 1.041, "EV": 1.011, "CR": 1.0}'
JUDGE_MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

LOG_FILE_4A = f"logs/exp_4a-{RUN_TAG}.log"
NOHUP_LOG_4A = f"logs/nohup-4a-{RUN_TAG}.log"
SUMMARY_4A = f"results/ex4a-cbtdp-dpo-{RUN_TAG}/summary_en.md"
CBTDP_DIR = f"models/cbtdp-dpo-llama3.1-8b-lora-{RUN_TAG}"
print("4A log:    ", LOG_FILE_4A)
print("4A summary:", SUMMARY_4A)
print("Adapter (shared with 4B):", CBTDP_DIR)

## 2A. Launch 4A in the background

In [ ]:
import subprocess, os

env = os.environ.copy()
env["RUN_TAG"] = RUN_TAG
env["SKILL_WEIGHTS"] = SKILL_WEIGHTS
env["JUDGE_MODEL"] = JUDGE_MODEL

os.makedirs("logs", exist_ok=True)
with open(NOHUP_LOG_4A, "w") as logf:
    proc = subprocess.Popen(
        ["nohup", "bash", "run_exp_4a.sh"],
        stdin=subprocess.DEVNULL, stdout=logf, stderr=subprocess.STDOUT,
        env=env, start_new_session=True,
    )
with open(f".run_4a_{RUN_TAG}.pid", "w") as f:
    f.write(str(proc.pid))
print(f"Launched run_exp_4a.sh, PID={proc.pid} -- saved to .run_4a_{RUN_TAG}.pid")

## 3A. Check on it
Re-run any time, even in a fresh kernel.

In [ ]:
import subprocess

def is_running(pid):
    return subprocess.run(f"kill -0 {pid}", shell=True).returncode == 0

try:
    with open(f".run_4a_{RUN_TAG}.pid") as f:
        pid4a = int(f.read().strip())
    print(f"PID {pid4a}: {'RUNNING' if is_running(pid4a) else 'not running (finished or was killed)'}")
    subprocess.run(f"ps -o pid,etime,cmd -p {pid4a}", shell=True)
except FileNotFoundError:
    print("No PID file yet -- run the launch cell first.")

## 4A. Tail the log

In [ ]:
import subprocess
print(f"--- last 60 lines of {LOG_FILE_4A} ---")
subprocess.run(f"tail -n 60 {LOG_FILE_4A} 2>/dev/null || echo '(not written yet)'", shell=True)
print(f"\n--- last 20 lines of {NOHUP_LOG_4A} (launch-time errors, if any) ---")
subprocess.run(f"tail -n 20 {NOHUP_LOG_4A} 2>/dev/null || echo '(empty)'", shell=True)

## 5A. GPU usage

In [ ]:
import subprocess
subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", shell=True)

## 6A. Read the 4A summary once done

In [ ]:
import subprocess, os
if os.path.exists(SUMMARY_4A):
    subprocess.run(f"cat {SUMMARY_4A}", shell=True)
else:
    print(f"Not written yet: {SUMMARY_4A}")
    print("Check cell 4A's log for '== 6/6 ==' or 'Done. Summary written to ...' before expecting this.")

---
# Part 2: Experiment 4B (reuse the SAME adapter, test on Arabic Shifaa)

**Do not run this section until 4A's log (cell 4A) shows the final `Done.` line and cell 6A
prints a real summary.** The launch cell below checks for the adapter directory and refuses
to run if it isn't there yet.

## 1B. Configure (reuses RUN_TAG/JUDGE_MODEL from Part 1 -- no need to redefine)

In [ ]:
LOG_FILE_4B = f"logs/exp_4-{RUN_TAG}.log"
NOHUP_LOG_4B = f"logs/nohup-4b-{RUN_TAG}.log"
SUMMARY_4B = f"results/ex4-cbtdp-dpo-{RUN_TAG}/summary_shifaa.md"
print("4B log:    ", LOG_FILE_4B)
print("4B summary:", SUMMARY_4B)
print("Reusing adapter:", CBTDP_DIR)

## 2B. Launch 4B in the background (SKIP_TRAIN=1 -- no retraining)

In [ ]:
import subprocess, os

if not os.path.isdir(CBTDP_DIR):
    raise RuntimeError(
        f"{CBTDP_DIR} does not exist yet -- 4A hasn't finished training. "
        "Run and wait for Part 1 first."
    )

env = os.environ.copy()
env["RUN_TAG"] = RUN_TAG
env["SKIP_TRAIN"] = "1"
env["CBTDP_DIR_OVERRIDE"] = CBTDP_DIR
env["JUDGE_MODEL"] = JUDGE_MODEL

with open(NOHUP_LOG_4B, "w") as logf:
    proc = subprocess.Popen(
        ["nohup", "bash", "run_exp_4.sh"],
        stdin=subprocess.DEVNULL, stdout=logf, stderr=subprocess.STDOUT,
        env=env, start_new_session=True,
    )
with open(f".run_4b_{RUN_TAG}.pid", "w") as f:
    f.write(str(proc.pid))
print(f"Launched run_exp_4.sh (SKIP_TRAIN=1), PID={proc.pid} -- saved to .run_4b_{RUN_TAG}.pid")

## 3B. Check on it

In [ ]:
import subprocess

def is_running(pid):
    return subprocess.run(f"kill -0 {pid}", shell=True).returncode == 0

try:
    with open(f".run_4b_{RUN_TAG}.pid") as f:
        pid4b = int(f.read().strip())
    print(f"PID {pid4b}: {'RUNNING' if is_running(pid4b) else 'not running (finished or was killed)'}")
    subprocess.run(f"ps -o pid,etime,cmd -p {pid4b}", shell=True)
except FileNotFoundError:
    print("No PID file yet -- run the launch cell first.")

## 4B. Tail the log

In [ ]:
import subprocess
print(f"--- last 60 lines of {LOG_FILE_4B} ---")
subprocess.run(f"tail -n 60 {LOG_FILE_4B} 2>/dev/null || echo '(not written yet)'", shell=True)
print(f"\n--- last 20 lines of {NOHUP_LOG_4B} ---")
subprocess.run(f"tail -n 20 {NOHUP_LOG_4B} 2>/dev/null || echo '(empty)'", shell=True)

## 5B. GPU usage

In [ ]:
import subprocess
subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", shell=True)

## 6B. Read the 4B summary once done

In [ ]:
import subprocess, os
if os.path.exists(SUMMARY_4B):
    subprocess.run(f"cat {SUMMARY_4B}", shell=True)
else:
    print(f"Not written yet: {SUMMARY_4B}")

## Notes

- **If the kernel or browser disconnects**, both jobs keep running on the server (`nohup` +
  `start_new_session=True`). Reopen this notebook, re-run cell 1A (config, harmless -- just
  sets variables), then whichever check/tail/summary cells you need -- no relaunch needed.
- **Never re-run a launch cell (2A or 2B) for a job that's already running or already
  finished** -- it starts a second, duplicate process under the same RUN_TAG, racing on the
  same output files. Check cell 3A/3B first if unsure.
- **To try a different config** (new skill weights, new judge model): change `RUN_TAG` in
  cell 1A to something new, then run everything from there again -- a new `RUN_TAG` never
  touches an earlier run's files.
- Same n-size caution as always: 4A tests on 32 items, 4B on whatever `N` Shifaa questions
  (default 50) -- read both as directional data points, not final verdicts.